In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "oss_ecosystem"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_github_events"

EVENT_TYPES = [
    "PushEvent",
    "PullRequestEvent",
    "IssuesEvent",
    "IssueCommentEvent",
    "WatchEvent",
    "ForkEvent",
    "ReleaseEvent"
]

In [0]:
payload_keys_df = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("event_type").isin(EVENT_TYPES))
    .select(
        "event_type",
        "source_hour",
        F.explode(
            F.json_object_keys("payload_json")
        ).alias("payload_key")
    )
)

payload_keys_df.show(20, truncate=False)

In [0]:
hourly_payload_schema_df = (
    payload_keys_df
    .groupBy(
        "event_type",
        "source_hour"
    )
    .agg(
        F.array_sort(
            F.collect_set("payload_key")
        ).alias("payload_keys")
    )
    .withColumn(
        "schema_signature",
        F.sha2(
            F.concat_ws("|", F.col("payload_keys")),
            256
        )
    )
)

hourly_payload_schema_df.orderBy(
    "event_type",
    "source_hour"
).show(100, truncate=False)

In [0]:
schema_variation_summary_df = (
    hourly_payload_schema_df
    .groupBy("event_type")
    .agg(
        F.countDistinct("schema_signature")
            .alias("distinct_hourly_schemas"),

        F.countDistinct("source_hour")
            .alias("hours_observed")
    )
    .orderBy(
        F.desc("distinct_hourly_schemas"),
        "event_type"
    )
)

schema_variation_summary_df.show(
    truncate=False
)

In [0]:
row_schema_df = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("event_type").isin(EVENT_TYPES))
    .withColumn(
        "payload_keys",
        F.array_sort(
            F.json_object_keys("payload_json")
        )
    )
    .withColumn(
        "schema_signature",
        F.sha2(
            F.concat_ws("|", F.col("payload_keys")),
            256
        )
    )
)

In [0]:
row_schema_summary_df = (
    row_schema_df
    .groupBy(
        "event_type",
        "schema_signature",
        "payload_keys"
    )
    .agg(
        F.count("*").alias("row_count")
    )
    .orderBy(
        "event_type",
        F.desc("row_count")
    )
)

row_schema_summary_df.show(
    100,
    truncate=False
)

In [0]:
row_schema_count_df = (
    row_schema_df
    .groupBy("event_type")
    .agg(
        F.countDistinct("schema_signature")
            .alias("distinct_row_schemas")
    )
    .orderBy(
        F.desc("distinct_row_schemas"),
        "event_type"
    )
)

row_schema_count_df.show(
    truncate=False
)

In [0]:
row_schema_summary_df.filter(
    F.col("event_type") == "IssuesEvent"
).show(
    20,
    truncate=False
)

In [0]:
row_schema_summary_df.filter(
    F.col("event_type") == "PullRequestEvent"
).show(
    20,
    truncate=False
)

In [0]:
issues_nested_keys_df = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("event_type") == "IssuesEvent")
    .withColumn(
        "issue_json",
        F.get_json_object("payload_json", "$.issue")
    )
    .withColumn(
        "issue_keys",
        F.array_sort(
            F.json_object_keys("issue_json")
        )
    )
    .groupBy("issue_keys")
    .count()
    .orderBy(F.desc("count"))
)

issues_nested_keys_df.show(
    50,
    truncate=False
)

In [0]:
pr_nested_keys_df = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("event_type") == "PullRequestEvent")
    .withColumn(
        "pr_json",
        F.get_json_object("payload_json", "$.pull_request")
    )
    .withColumn(
        "pr_keys",
        F.array_sort(
            F.json_object_keys("pr_json")
        )
    )
    .groupBy("pr_keys")
    .count()
    .orderBy(F.desc("count"))
)

pr_nested_keys_df.show(
    50,
    truncate=False
)

In [0]:
schema_baseline_df = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("event_type").isin(EVENT_TYPES))
    .select(
        "event_type",
        F.explode(
            F.json_object_keys("payload_json")
        ).alias("payload_key")
    )
    .distinct()
    .groupBy("event_type")
    .agg(
        F.array_sort(
            F.collect_set("payload_key")
        ).alias("expected_payload_keys")
    )
    .orderBy("event_type")
)

schema_baseline_df.show(
    truncate=False
)

In [0]:
schema_baseline_df.select(
    "event_type",
    "expected_payload_keys"
).show(
    truncate=False
)

In [0]:
BASELINE_TABLE = f"{CATALOG}.{SCHEMA}.payload_schema_baseline"

if not spark.catalog.tableExists(BASELINE_TABLE):
    (
        schema_baseline_df
        .withColumn(
            "baseline_created_at",
            F.current_timestamp()
        )
        .write
        .format("delta")
        .saveAsTable(BASELINE_TABLE)
    )

    print("Schema baseline created.")
else:
    print("Schema baseline already exists; not overwritten.")

In [0]:
observed_schema_df = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("event_type").isin(EVENT_TYPES))
    .select(
        "event_type",
        F.explode(
            F.json_object_keys("payload_json")
        ).alias("payload_key")
    )
    .distinct()
    .groupBy("event_type")
    .agg(
        F.array_sort(
            F.collect_set("payload_key")
        ).alias("observed_payload_keys")
    )
)

In [0]:
schema_validation_df = (
    observed_schema_df.alias("o")
    .join(
        spark.table(BASELINE_TABLE).alias("b"),
        on="event_type",
        how="left"
    )
    .withColumn(
        "unexpected_keys",
        F.array_except(
            F.col("observed_payload_keys"),
            F.col("expected_payload_keys")
        )
    )
    .withColumn(
        "has_unexpected_keys",
        F.size("unexpected_keys") > 0
    )
    .select(
        "event_type",
        "expected_payload_keys",
        "observed_payload_keys",
        "unexpected_keys",
        "has_unexpected_keys"
    )
)

schema_validation_df.show(
    truncate=False
)

In [0]:
schema_drift_test_df = (
    schema_validation_df
    .filter(F.col("event_type") == "IssuesEvent")
    .withColumn(
        "observed_payload_keys",
        F.array_union(
            F.col("observed_payload_keys"),
            F.array(F.lit("synthetic_new_field"))
        )
    )
    .withColumn(
        "unexpected_keys",
        F.array_except(
            F.col("observed_payload_keys"),
            F.col("expected_payload_keys")
        )
    )
    .withColumn(
        "has_unexpected_keys",
        F.size("unexpected_keys") > 0
    )
)

schema_drift_test_df.select(
    "event_type",
    "unexpected_keys",
    "has_unexpected_keys"
).show(truncate=False)

In [0]:
required_payload_keys = {
    "PushEvent": [
        "push_id",
        "ref",
        "head",
        "before"
    ],

    "PullRequestEvent": [
        "action",
        "number",
        "pull_request"
    ],

    "IssuesEvent": [
        "action",
        "issue"
    ],

    "IssueCommentEvent": [
        "action",
        "issue",
        "comment"
    ],

    "WatchEvent": [
        "action"
    ],

    "ForkEvent": [
        "forkee"
    ],

    "ReleaseEvent": [
        "action",
        "release"
    ]
}

In [0]:
required_schema_df = spark.createDataFrame(
    [
        (event_type, keys)
        for event_type, keys in required_payload_keys.items()
    ],
    [
        "event_type",
        "required_payload_keys"
    ]
)

required_key_validation_df = (
    observed_schema_df.alias("o")
    .join(
        required_schema_df.alias("r"),
        on="event_type",
        how="inner"
    )
    .withColumn(
        "missing_required_keys",
        F.array_except(
            F.col("required_payload_keys"),
            F.col("observed_payload_keys")
        )
    )
    .withColumn(
        "has_missing_required_keys",
        F.size("missing_required_keys") > 0
    )
    .select(
        "event_type",
        "required_payload_keys",
        "observed_payload_keys",
        "missing_required_keys",
        "has_missing_required_keys"
    )
    .orderBy("event_type")
)

required_key_validation_df.show(
    truncate=False
)

In [0]:
missing_key_test_df = (
    required_key_validation_df
    .filter(
        F.col("event_type") == "PullRequestEvent"
    )
    .withColumn(
        "observed_payload_keys",
        F.array_remove(
            F.col("observed_payload_keys"),
            "pull_request"
        )
    )
    .withColumn(
        "missing_required_keys",
        F.array_except(
            F.col("required_payload_keys"),
            F.col("observed_payload_keys")
        )
    )
    .withColumn(
        "has_missing_required_keys",
        F.size("missing_required_keys") > 0
    )
)

missing_key_test_df.select(
    "event_type",
    "missing_required_keys",
    "has_missing_required_keys"
).show(
    truncate=False
)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, LongType

pr_validation_schema = StructType([
    StructField("action", StringType(), True),
    StructField("number", LongType(), True),
    StructField(
        "pull_request",
        StructType([
            StructField("id", LongType(), True),
            StructField("number", LongType(), True),
            StructField("url", StringType(), True)
        ]),
        True
    )
])

pr_parse_validation_df = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("event_type") == "PullRequestEvent")
    .withColumn(
        "parsed_payload",
        F.from_json(
            F.col("payload_json"),
            pr_validation_schema
        )
    )
    .withColumn(
        "pull_request_key_present",
        F.array_contains(
            F.json_object_keys("payload_json"),
            "pull_request"
        )
    )
    .withColumn(
        "pull_request_parse_failed",
        F.col("pull_request_key_present")
        & F.col("parsed_payload.pull_request").isNull()
    )
)

pr_parse_validation_df.select(
    F.count("*").alias("total_pr_events"),

    F.sum(
        F.col("pull_request_parse_failed").cast("int")
    ).alias("parse_failures")
).show()

In [0]:
synthetic_bad_payload_df = spark.createDataFrame(
    [
        (
            '{"action":"opened","number":123,"pull_request":"not_an_object"}',
        )
    ],
    ["payload_json"]
)

synthetic_type_test_df = (
    synthetic_bad_payload_df
    .withColumn(
        "parsed_payload",
        F.from_json(
            F.col("payload_json"),
            pr_validation_schema
        )
    )
    .withColumn(
        "pull_request_key_present",
        F.array_contains(
            F.json_object_keys("payload_json"),
            "pull_request"
        )
    )
    .withColumn(
        "pull_request_parse_failed",
        F.col("pull_request_key_present")
        & F.col("parsed_payload.pull_request").isNull()
    )
)

synthetic_type_test_df.select(
    "payload_json",
    "pull_request_key_present",
    "pull_request_parse_failed"
).show(
    truncate=False
)

## Phase 6 conclusions

Observed schema variation:

- IssuesEvent has 3 top-level payload schemas.
- PullRequestEvent has 3 top-level payload schemas.
- Other modeled event types currently have 1 top-level payload schema.
- IssuesEvent also shows nested variation inside the issue object.
- The Silver parsers intentionally consume only the stable fields required by downstream models.

Schema controls implemented:

- Baseline of known payload keys stored in Delta.
- Detection of unexpected additive fields.
- Detection of missing required fields.
- Detection of incompatible/unparseable required nested structures.
- Baseline is not automatically overwritten when new fields appear.

Design decision:

Schema evolution is handled explicitly rather than by blindly enabling automatic schema merge.
Optional source fields are tolerated, while changes affecting required Silver fields are flagged for review.